# Part 1 Extra: Advanced Federated Learning Strategies (FedProx & FedAdam)

This notebook tests what happens when we replace standard `FedAvg` with advanced aggregation strategies.

## 1. Setup and Preprocessing (Same as before)

In [ ]:
import pandas as pd
import numpy as np
import threading
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import flwr as fl
from typing import Dict, List, Optional, Tuple
from flwr.common import Metrics

print("Imports DDone")

## 2. Advanced FL Simulations (FedProx vs FedAvg vs FedAdam)

*Note on FedProx: In a true PyTorch implementation, the hospital client adds a mathematical penalty (mu) to its loss function. Since we are using Scikit-Learn (which has hardcoded C-optimizers), we configure the Flower Server for FedProx, but the clients will behave similarly to FedAvg. 

*Note on FedAdam: This is a purely server-side optimization. The server applies the Adam optimizer to the incoming weights!

In [ ]:
def get_model_params(model):
    if hasattr(model, "coef_"):
        return [model.coef_, model.intercept_]
    return []

def set_model_params(model, parameters):
    if hasattr(model, "coef_"):
        model.coef_ = parameters[0]
        model.intercept_ = parameters[1]
    if not hasattr(model, "classes_"):
        model.classes_ = np.array([0, 1])

class HeartDiseaseClient(fl.client.NumPyClient):
    def __init__(self, model, X_train, y_train, X_test, y_test):
        self.model = model
        self.X_train = X_train
        self.y_train = y_train
        self.X_test = X_test
        self.y_test = y_test

    def get_parameters(self, config):
        return get_model_params(self.model)

    def fit(self, parameters, config):
        set_model_params(self.model, parameters)
        self.model.fit(self.X_train, self.y_train)
        return self.get_parameters(config={}), len(self.X_train), {}

    def evaluate(self, parameters, config):
        set_model_params(self.model, parameters)
        y_pred = self.model.predict(self.X_test)
        accuracy = accuracy_score(self.y_test, y_pred)
        return float(1.0), len(self.X_test), {"accuracy": float(accuracy)}

def evaluate_metrics_aggregation_fn(metrics: List[Tuple[int, Metrics]]) -> Metrics:
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]
    return {"accuracy": sum(accuracies) / sum(examples)}

# Read n_features dynamically to prevent shape mismatch in FedAdam
temp_df = pd.read_csv("data/processed_data.csv")
n_features = temp_df.shape[1] - 1  # subtract target

strategies = {
    "FedAvg": fl.server.strategy.FedAvg(
        fraction_fit=1.0, fraction_evaluate=1.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
        evaluate_metrics_aggregation_fn=evaluate_metrics_aggregation_fn,
    ),
    "FedProx (mu=1.0)": fl.server.strategy.FedProx(
        fraction_fit=1.0, fraction_evaluate=1.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
        evaluate_metrics_aggregation_fn=evaluate_metrics_aggregation_fn,
        proximal_mu=1.0
    ),
    "FedAdam": fl.server.strategy.FedAdam(
        fraction_fit=1.0, fraction_evaluate=1.0,
        min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4,
        evaluate_metrics_aggregation_fn=evaluate_metrics_aggregation_fn,
        initial_parameters=fl.common.ndarrays_to_parameters([np.zeros((1, n_features)), np.zeros(1)])
    )
}

fl_results_history = {}
PORT = 9080

for strat_name, strategy_obj in strategies.items():
    print(f"\n{'='*40}\nRunning Simulation using: {strat_name}\n{'='*40}")
    PORT += 1
    
    global_metrics = {"round": [], "accuracy": []}
    
    class MetricsCaptureStrategy(type(strategy_obj)):
        def aggregate_evaluate(self, server_round, results, failures):
            loss, metrics = super().aggregate_evaluate(server_round, results, failures)
            if loss is not None and metrics is not None:
                global_metrics["round"].append(server_round)
                global_metrics["accuracy"].append(metrics["accuracy"])
            return loss, metrics

    if strat_name == "FedProx (mu=1.0)":
        active_strategy = MetricsCaptureStrategy(fraction_fit=1.0, fraction_evaluate=1.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, evaluate_metrics_aggregation_fn=evaluate_metrics_aggregation_fn, proximal_mu=1.0)
    elif strat_name == "FedAdam":
        active_strategy = MetricsCaptureStrategy(fraction_fit=1.0, fraction_evaluate=1.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, evaluate_metrics_aggregation_fn=evaluate_metrics_aggregation_fn, initial_parameters=fl.common.ndarrays_to_parameters([np.zeros((1, n_features)), np.zeros(1)]))
    else:
        active_strategy = MetricsCaptureStrategy(fraction_fit=1.0, fraction_evaluate=1.0, min_fit_clients=4, min_evaluate_clients=4, min_available_clients=4, evaluate_metrics_aggregation_fn=evaluate_metrics_aggregation_fn)

    def run_server():
        fl.server.start_server(server_address=f"127.0.0.1:{PORT}", config=fl.server.ServerConfig(num_rounds=5), strategy=active_strategy)

    def run_client(cid):
        client_id = int(cid) + 1
        df = pd.read_csv(f"data/hospital_{client_id}/train.csv")
        X = df.drop("target", axis=1).values
        y = df["target"].values
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        model = LogisticRegression(max_iter=1, warm_start=True, solver="saga", random_state=42)
        model.classes_ = np.array([0, 1])
        model.coef_ = np.zeros((1, X_train.shape[1]))
        model.intercept_ = np.zeros(1)
        
        client = HeartDiseaseClient(model, X_train, y_train, X_test, y_test).to_client()
        fl.client.start_client(server_address=f"127.0.0.1:{PORT}", client=client)

    def delayed_start_clients():
        time.sleep(3)
        client_threads = []
        for i in range(4):
            t = threading.Thread(target=run_client, args=(i,))
            t.start()
            client_threads.append(t)
        for t in client_threads:
            t.join()

    client_launcher_thread = threading.Thread(target=delayed_start_clients)
    client_launcher_thread.start()
    run_server()
    client_launcher_thread.join()
    
    fl_results_history[strat_name] = global_metrics

print("\nAll Strategy Simulations Finished!")

## 3. Compare the Strategies

In [ ]:
plt.figure(figsize=(10, 6))
colors = ['blue', 'green', 'red']
for (strat_name, metrics), color in zip(fl_results_history.items(), colors):
    plt.plot(metrics['round'], np.array(metrics['accuracy']) * 100, marker='o', label=f'Strategy: {strat_name}', color=color)
    
plt.title('Federated Learning Strategy Comparison (Logistic Regression)')
plt.xlabel('Communication Round')
plt.ylabel('Accuracy (%)')
plt.ylim(0, 100)
plt.xticks(range(1, 6))
plt.legend()
plt.grid(True)
plt.show()